# Apply SinSpeech manual-listening QA to OpenSLR-52

Applies the human-verified corrections from `SinSpeech-Development/Corpus-Cleaning`'s
`Sinhala_Rules_Applied/` stage to `utt_spk_text_cleaned.tsv` (the output of
`transcript_openslr.ipynb`).

**Scope -- deliberately narrow.** That repo does two stages:

1. `OpenSLR-52-Sinhala/` -- automated Unicode/punctuation cleanup, including a
   rakaranshaya/yansaya (ZWJ) reconstruction. Investigation showed this stage
   *creates* its own defect first (`str.replace("\u200D", "")` strips **all**
   zero-width joiners, including ~27,800 legitimate ones already correct in the raw
   OpenSLR TSV) and then partially restores them via a hand-built dictionary of
   ~1,582 words. `clean_hidden_chars.py`/`check_hidden_chars.py` in this project
   already do this correctly -- classifying each ZWJ by its actual neighbors in the
   original string instead of blanket-stripping -- so that stage is **not** reused
   here.
2. `Sinhala_Rules_Applied/` -- a second, manual stage: a person listened to
   ambiguous vowel-boundary utterances and produced `Corrected/`, `Doubt_Corrected/`,
   and `To_Delete/` verdicts per utterance/word. This is real audio-verified labor
   with no equivalent in this project's pipeline. **This notebook applies only this
   stage.**

Three QA artifacts, all keyed to the original OpenSLR `utterance_id` (== this
project's `file_id`):

- `Corrected/corrected_*.csv` + `common_modifications_{nilmani,disura}.csv` --
  global word/phrase corrections (`original` -> `correction`), found while
  reviewing a specific ambiguous pattern but applicable wherever the word occurs.
- `Doubt_Corrected/doubtCorrected_*.csv` -- corrections scoped to specific
  `utterance_ids` (edge cases that needed listening to confirm, not a safe global
  substitution).
- `To_Delete/toDelete_*.csv` -- utterance ids SinSpeech identified as unusable
  after listening.

**No rows are dropped in this notebook.** `transcript_openslr.ipynb` only filters
out *pure*-English rows (no Sinhala script at all) -- code-mixed Sinhala/English
utterances are intentionally kept. `To_Delete` ids are only *flagged* here (logged
to `qa_flagged_ids.csv` for your own review) so you can decide per-row later; every
row of `utt_spk_text_cleaned.tsv` survives into `utt_spk_text_qa.tsv`, only `text`
is ever changed.

In [1]:
import glob
import os
import re

import pandas as pd

QA_REPO = "../../../../openslrTranscription/Corpus-Cleaning/Sinhala_Rules_Applied"
CLEAN_DIR = "../../../../data/raw/openslr_52/processed/openslr_52_clean"
TSV_IN = os.path.join(CLEAN_DIR, "utt_spk_text_cleaned.tsv")
TSV_OUT = os.path.join(CLEAN_DIR, "utt_spk_text_qa.tsv")
CHANGES_LOG = os.path.join(CLEAN_DIR, "qa_changes_log.csv")
FLAGGED_LOG = os.path.join(CLEAN_DIR, "qa_flagged_ids.csv")
CONFLICTS_LOG = os.path.join(CLEAN_DIR, "qa_conflicting_corrections_SKIPPED.csv")

## Step 1 -- Global word/phrase corrections

Merges `Corrected/corrected_*.csv` (64 files, one per reviewed pattern) with both
`common_modifications_*.csv` dictionaries. About half of `Corrected/*.csv` rows are
no-ops (`original == correction`, i.e. "reviewed, found already correct") -- those
are dropped, not applied.

**Conflict check:** the same `original` string occasionally maps to two different
corrections across files (e.g. `අදාලද` -> `අදාල ද` in one file,
`අදාල ද` in another -- an ල/ළ spelling-convention disagreement between
reviewers). Applying either guess automatically risks being wrong for context this
notebook can't see, so any `original` key with more than one distinct `correction`
across the merged set is **excluded** from auto-apply and logged to
`qa_conflicting_corrections_SKIPPED.csv` for manual review instead.

In [2]:
corr_frames = [
    pd.read_csv(f, dtype=str, keep_default_na=False)
    for f in glob.glob(f"{QA_REPO}/Corrected/corrected_*.csv")
]
corr_all = pd.concat(corr_frames, ignore_index=True)[["original", "correction"]]
real = corr_all[corr_all["original"] != corr_all["correction"]]

nilmani = pd.read_csv(f"{QA_REPO}/common_modifications_nilmani.csv", dtype=str)[["original", "correction"]]
disura = pd.read_csv(f"{QA_REPO}/common_modifications_disura.csv", dtype=str)[["original", "correction"]]

allwords = pd.concat([real, nilmani, disura], ignore_index=True).drop_duplicates()

conflict_counts = allwords.groupby("original")["correction"].nunique()
conflict_keys = set(conflict_counts[conflict_counts > 1].index)
allwords[allwords["original"].isin(conflict_keys)].sort_values("original").to_csv(CONFLICTS_LOG, index=False)

safe = allwords[~allwords["original"].isin(conflict_keys)].drop_duplicates("original")
word_map = dict(zip(safe["original"], safe["correction"]))

print(f"{len(word_map)} safe word/phrase corrections, {len(conflict_keys)} conflicting keys skipped -> {CONFLICTS_LOG}")

9327 safe word/phrase corrections, 38 conflicting keys skipped -> ../../../../data/raw/openslr_52/processed/openslr_52_clean/qa_conflicting_corrections_SKIPPED.csv


Corrections are applied as **whole-word/phrase matches**, not raw substring
replacement -- `original` is anchored with `(?<!\S)...(?!\S)` (whitespace
boundaries on both sides) so a 2-character key like `ඉද` can never match
inside the middle of a longer, unrelated word. Some `original` entries are
themselves multi-word phrases (e.g. pair-review outputs), so the pattern is a single
alternation over all keys, longest first, rather than per-token matching.

In [3]:
keys_sorted = sorted(word_map, key=len, reverse=True)
word_pattern = re.compile(r'(?<!\S)(' + '|'.join(re.escape(k) for k in keys_sorted) + r')(?!\S)')


def apply_word_corrections(text):
    return word_pattern.sub(lambda m: word_map[m.group(0)], text)

## Step 2 -- Id-scoped corrections

`Doubt_Corrected/*.csv` rows are edge cases a reviewer had to listen to confirm --
each is scoped to a specific `utterance_ids` value rather than applied globally.

In [4]:
doubt_frames = [
    pd.read_csv(f, dtype=str, keep_default_na=False)
    for f in glob.glob(f"{QA_REPO}/Doubt_Corrected/doubtCorrected_*.csv")
]
doubt_all = pd.concat(doubt_frames, ignore_index=True)
doubt_all = doubt_all[doubt_all["original"] != doubt_all["correction"]]

id_corrections = {}
for _, row in doubt_all.iterrows():
    pat = re.compile(r'(?<!\S)' + re.escape(row["original"]) + r'(?!\S)')
    id_corrections.setdefault(row["utterance_ids"], []).append((pat, row["correction"]))

print(f"{len(id_corrections)} utterance ids have an id-scoped correction")

61 utterance ids have an id-scoped correction


## Step 3 -- Utterances SinSpeech flagged as unusable

`To_Delete/*.csv` (24 files) -- utterance ids SinSpeech's reviewers dropped after
listening. **Not dropped here** -- only flagged/logged, per instruction to keep
every row (including code-mixed Sinhala/English ones, which this project's own
pipeline already deliberately kept). Review `qa_flagged_ids.csv` yourself if you
want to drop any of these later.

In [5]:
del_frames = [pd.read_csv(f, dtype=str) for f in glob.glob(f"{QA_REPO}/To_Delete/toDelete_*.csv")]
flagged_ids = set(pd.concat(del_frames)["utterance_id"])
print(f"{len(flagged_ids)} utterance ids flagged by SinSpeech (not dropped)")

313 utterance ids flagged by SinSpeech (not dropped)


## Load your cleaned corpus and check coverage

`utt_spk_text_cleaned.tsv` is the output of `transcript_openslr.ipynb` -- already
past this project's own NFC/whitespace/punctuation/emoji/whitelist/ZWJ cleaning, and
already missing ~28k rows that were in SinSpeech's 178,030-row corpus but got
dropped independently by your pipeline or the upstream VAD/dedup stage. So only a
fraction of SinSpeech's flagged ids/words will still be present here -- that's
expected, not a bug.

In [6]:
df = pd.read_csv(TSV_IN, sep="\t", dtype=str, keep_default_na=False)
print(f"loaded {len(df)} rows from {TSV_IN}")

clean_ids = set(df["file_id"])
print(f"\nflagged ids present in your corpus:      {len(flagged_ids & clean_ids)} / {len(flagged_ids)}")
print(f"id-scoped correction ids present:        {len(set(id_corrections) & clean_ids)} / {len(id_corrections)}")

loaded 150191 rows from ../../../../data/raw/openslr_52/processed/openslr_52_clean/utt_spk_text_cleaned.tsv

flagged ids present in your corpus:      265 / 313
id-scoped correction ids present:        42 / 61


## Apply corrections -- no rows dropped

Every row of `utt_spk_text_cleaned.tsv` is kept. Flagged ids are logged (for your
own optional review) but left in the corpus with their text unchanged. Id-scoped
corrections are applied before global ones -- they're the more specific,
reviewer-confirmed fix for that exact utterance; the global pass then still covers
any other words in the same row.

In [7]:
flagged_mask = df["file_id"].isin(flagged_ids)
df.loc[flagged_mask, ["file_id", "speaker_id", "text"]].to_csv(FLAGGED_LOG, index=False)
print(f"{flagged_mask.sum()} rows flagged (kept, not dropped) -> {FLAGGED_LOG}")

265 rows flagged (kept, not dropped) -> ../../../../data/raw/openslr_52/processed/openslr_52_clean/qa_flagged_ids.csv


In [8]:
def apply_id_scoped(row):
    fid, text = row["file_id"], row["text"]
    if fid in id_corrections:
        for pat, corr in id_corrections[fid]:
            text = pat.sub(corr, text)
    return text


text_before = df["text"].copy()
df["text"] = df.apply(apply_id_scoped, axis=1)
df["text"] = df["text"].apply(apply_word_corrections)
df["text_len"] = df["text"].str.len()

changed_mask = df["text"] != text_before
pd.DataFrame({
    "file_id": df.loc[changed_mask, "file_id"],
    "before": text_before[changed_mask],
    "after": df.loc[changed_mask, "text"],
}).to_csv(CHANGES_LOG, index=False)

print(f"changed {changed_mask.sum()} / {len(df)} rows ({changed_mask.mean():.1%}) -> {CHANGES_LOG}")

changed 31086 / 150191 rows (20.7%) -> ../../../../data/raw/openslr_52/processed/openslr_52_clean/qa_changes_log.csv


## Verify -- sample before/after

In [9]:
sample = df[changed_mask].sample(min(10, changed_mask.sum()), random_state=1)
for _, r in sample.iterrows():
    before = text_before.loc[r.name]
    print(f"[{r['file_id']}]")
    print(f"  before: {before!r}")
    print(f"  after:  {r['text']!r}")
    print()

[e0f25433ae]
  before: 'තරුණ කාන්තාවන් සමග සල්ලාලකම් වල යෙදෙති.'
  after:  'තරුණ කාන්තාවන් සමග සල්ලාලකම්වල යෙදෙති.'

[d6f01a8a02]
  before: 'සැබවින්ම පංති විරහිත සමාජ ක්\u200dරමයක්'
  after:  'සැබවින් ම පංති විරහිත සමාජ ක්\u200dරමයක්'

[e31380009a]
  before: 'අද අපට මුහුණ දීමට සිදුවන මූලිකම අභියෝගයක් තමයි'
  after:  'අද අපට මුහුණ දීමට සිදු වන මූලික ම අභියෝගයක් තමයි'

[0ee2dfaef6]
  before: 'අනුර කියනව පරදිනව කියන එක'
  after:  'අනුර කියනවා පරදිනවා කියන එක'

[ec4d9903c9]
  before: 'සැවොම එක්ව බ්ලොග් කියවන්න'
  after:  'සැවොම එක් ව බ්ලොග් කියවන්න'

[eb3254a701]
  before: 'අද සිදුවන හැම ක්\u200dරියාවක්ම'
  after:  'අද සිදු වන හැම ක්\u200dරියාවක් ම'

[bcb327e3d1]
  before: 'අවබෝධයෙන් යුතුව'
  after:  'අවබෝධයෙන් යුතු ව'

[fc737dbd23]
  before: 'ධර්මයෙහි අදත් ජීවමානව පවතින සූත්\u200dර ධර්ම'
  after:  'ධර්මයෙහි අදත් ජීවමාන ව පවතින සූත්\u200dර ධර්ම'

[a9b3a7a754]
  before: 'අදත් ඒ සර් අපිව හම්බුනාම'
  after:  'අදත් ඒ සර් අපිව හම්බුනා ම'

[65609d709f]
  before: 'අපි ඉවක් බවක් නැතිව විනාශ කරනවා 

## Save

Writes a **new** `utt_spk_text_qa.tsv` alongside `utt_spk_text_cleaned.tsv` -- does
not overwrite it, and does not touch any `.flac` file. Row count is asserted equal
to the input: this pass only ever rewrites `text`, never drops a row.

In [10]:
assert len(df) == len(pd.read_csv(TSV_IN, sep="\t", dtype=str, keep_default_na=False)), "row count changed -- should never happen in this notebook"

df.to_csv(TSV_OUT, sep="\t", index=False)
print(f"wrote {len(df)} rows -> {TSV_OUT}  (same row count as input -- nothing dropped)")

wrote 150191 rows -> ../../../../data/raw/openslr_52/processed/openslr_52_clean/utt_spk_text_qa.tsv  (same row count as input -- nothing dropped)


## Propagating into `final_dataset_openslr(.parquet)` and the training splits

**Not run automatically by this notebook -- read before running.**

`final_dataset_openslr.parquet` / `final_dataset_openslr_cleaned.parquet` /
`final_split_dataset/**/*.parquet` only carry `audio` (embedded WAV bytes),
`source_dataset`, `text` -- **no `file_id` column survives** past the parquet-export
step, so these QA corrections can't be joined onto the existing parquet files
directly. The only correct way to propagate them is to rebuild
`final_dataset_openslr.parquet` from `utt_spk_text_qa.tsv` the same way
`transcript_openslr.ipynb` built it the first time (re-reading and re-encoding every
`.flac` file to WAV), then re-run the two downstream steps that already exist in
this repo:

```
1. Rebuild final_dataset_openslr.parquet from utt_spk_text_qa.tsv  (cell below)
2. python3 dse-project/scripts/clean_hidden_chars.py \
       --openslr-in  dse-project/model-development/data/final_dataset/final_dataset_openslr_qa.parquet \
       --openslr-out dse-project/model-development/data/final_dataset/final_dataset_openslr_qa_cleaned.parquet
3. python3 dse-project/scripts/split_final_datasets.py \
       --openslr dse-project/model-development/data/final_dataset/final_dataset_openslr_qa_cleaned.parquet \
       --output-dir dse-project/model-development/data/final_split_dataset_qa
```

Step 1 re-encodes ~150k `.flac` files to WAV (same cost as the original
`convert_openslr.py`/`transcript_openslr.ipynb` run -- expect it to take a
meaningful amount of time and ~15GB of new disk) and steps 2-3 stream through that
output again. Output paths above are all suffixed `_qa`/`_qa_cleaned` so nothing
existing is overwritten; swap them into your training config once you've spot-checked
the result.

In [11]:
RUN_REBUILD = False  # flip to True (and run this cell) only when you're ready for the full re-encode

if RUN_REBUILD:
    import io
    import pyarrow as pa
    import pyarrow.parquet as pq
    import soundfile as sf

    FINAL_DIR = "../../data/final_dataset"
    OUT_PARQUET = os.path.join(FINAL_DIR, "final_dataset_openslr_qa.parquet")
    BATCH_SIZE = 2000

    schema = pa.schema([
        pa.field("audio", pa.binary()),
        pa.field("source_dataset", pa.string()),
        pa.field("text", pa.string()),
    ])

    def flac_to_wav_bytes(file_id):
        flac_path = os.path.join(CLEAN_DIR, "data", file_id[:2], f"{file_id}.flac")
        data, sr = sf.read(flac_path, dtype="int16", always_2d=False)
        buf = io.BytesIO()
        sf.write(buf, data, sr, format="WAV", subtype="PCM_16")
        return buf.getvalue()

    n_rows = len(df)
    writer = pq.ParquetWriter(OUT_PARQUET, schema)
    try:
        for start in range(0, n_rows, BATCH_SIZE):
            batch = df.iloc[start:start + BATCH_SIZE]
            audio_bytes = [flac_to_wav_bytes(fid) for fid in batch["file_id"]]
            table = pa.table(
                {
                    "audio": pa.array(audio_bytes, type=pa.binary()),
                    "source_dataset": pa.array(batch["source_dataset"], type=pa.string()),
                    "text": pa.array(batch["text"], type=pa.string()),
                },
                schema=schema,
            )
            writer.write_table(table)
            print(f"  wrote rows {start}-{start + len(batch)} ({start + len(batch)}/{n_rows})", end="\r")
    finally:
        writer.close()

    print(f"\nSaved {n_rows} rows -> {OUT_PARQUET} ({os.path.getsize(OUT_PARQUET) / 1e6:.1f} MB)")
    print("Next: run clean_hidden_chars.py and split_final_datasets.py as shown above.")
else:
    print("RUN_REBUILD is False -- skipped. Set to True when ready to re-encode audio and rebuild the parquet.")

RUN_REBUILD is False -- skipped. Set to True when ready to re-encode audio and rebuild the parquet.
